# MUC-4 baselines on a Colab T4

Zero-shot and 3-shot baselines of `Qwen/Qwen3-4B-Instruct-2507` on the dev split, each logged as one
W&B run in `muc4-event-extraction`. **Run all twice: the install cell restarts the runtime once.**

Needs the Colab Secrets `HF_TOKEN` and `WANDB_API_KEY` (key icon in the left sidebar). Set `REF` in
the clone cell to run a specific commit, branch or tag.

In [ ]:
import sys

import torch

!nvidia-smi
print("torch", torch.__version__, "| python", sys.version)

In [ ]:
import importlib.util

# Colab's torchaudio was built for the CUDA of the torch vLLM replaces; transformers imports it if present
!pip uninstall --yes --quiet torchaudio
if importlib.util.find_spec("vllm") is None:
    !pip install vllm==0.29.0 --extra-index-url https://wheels.vllm.ai/0.29.0/cu129 --extra-index-url https://download.pytorch.org/whl/cu129
    get_ipython().kernel.do_shutdown(restart=True)

In [ ]:
REF = "main"  # @param {type:"string"}
import os

%cd /content
if not os.path.isdir("fine-tuning-decoder"):
    !git clone --quiet https://github.com/fnl/fine-tuning-decoder.git
%cd /content/fine-tuning-decoder
!git fetch --quiet origin \
    && (git checkout --quiet --detach origin/$REF 2>/dev/null || git checkout --quiet --detach $REF) \
    && git log --oneline -1
!pip install --quiet -e .

In [ ]:
from importlib.metadata import version

import generate  # noqa: F401  (the editable install put src/ on the path)
import vllm

print("fine-tuning-decoder", version("fine-tuning-decoder"), "| vllm", vllm.__version__)

In [ ]:
import os

from google.colab import userdata

for name in ("HF_TOKEN", "WANDB_API_KEY"):
    value = userdata.get(name)
    assert value, f"{name} is missing from Colab Secrets"
    os.environ[name] = value

In [ ]:
import os

from datasets import load_dataset

os.makedirs("data/prepared", exist_ok=True)
load_dataset("fnl-es/muc4-chat")["dev"].to_json("data/prepared/dev.jsonl")

In [ ]:
!python -m generate --config configs/qwen3-4b-zero-shot.yaml --limit 5 --out outputs/smoke \
    && cat outputs/smoke/dev.jsonl

In [ ]:
!python -m generate --config configs/qwen3-4b-zero-shot.yaml \
    && python -m eval --config configs/qwen3-4b-zero-shot.yaml --pred outputs/qwen3-4b-zero-shot/dev.jsonl --gold data/prepared/dev.jsonl --wandb

In [ ]:
!python -m generate --config configs/qwen3-4b-3-shot.yaml \
    && python -m eval --config configs/qwen3-4b-3-shot.yaml --pred outputs/qwen3-4b-3-shot/dev.jsonl --gold data/prepared/dev.jsonl --wandb